# Epigenetic Mark Association Analysis
**Data**: `/api/v1/export/chipseq-overlaps`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
API_BASE_URL = 'http://localhost:8000/api/v1'
print('Setup complete')

In [ ]:
marks = ['H3K4me3', 'H3K27me3', 'H3K27ac', 'H3K4me1', 'H3K36me3', 'H3K9me3']
all_data = []
for m in marks:
    r = requests.get(f'{API_BASE_URL}/export/chipseq-overlaps', params={'mark_names': [m], 'min_ba': 100, 'limit': 10000})
    if r.status_code == 200:
        all_data.extend(r.json()['data'])
        print(f'{m}: {len(r.json()["data"])} records')
df = pd.DataFrame(all_data)
print(f'Total: {len(df)} records')

## Mark Distribution

In [ ]:
mark_counts = df['mark_name'].value_counts()
print(mark_counts)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
mark_counts.plot(kind='bar', ax=axes[0], edgecolor='black')
axes[0].set_title('ChIP-seq Overlaps by Mark', fontweight='bold')
axes[1].pie(mark_counts, labels=mark_counts.index, autopct='%1.1f%%')
axes[1].set_title('Mark Distribution', fontweight='bold')
plt.tight_layout()
plt.savefig('figures/09_chipseq_mark_distribution.png', dpi=300)
plt.show()

## Bivalent Domain Detection

In [ ]:
reg_marks = df.groupby('regulation_id')['mark_name'].apply(set).reset_index()
bivalent_mask = reg_marks['mark_name'].apply(lambda x: 'H3K4me3' in x and 'H3K27me3' in x)
bivalent_regs = reg_marks[bivalent_mask]['regulation_id'].tolist()
print(f'Bivalent domains: {len(bivalent_regs)} ({100*len(bivalent_regs)/len(reg_marks):.1f}%)')
biv_ba = df[df['regulation_id'].isin(bivalent_regs)].drop_duplicates('regulation_id')['binding_affinity']
non_ba = df[~df['regulation_id'].isin(bivalent_regs)].drop_duplicates('regulation_id')['binding_affinity']
u, p = stats.mannwhitneyu(biv_ba, non_ba)
print(f'Mann-Whitney U: p={p:.4e}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].hist([biv_ba, non_ba], bins=30, label=['Bivalent', 'Non-Bivalent'], alpha=0.7)
axes[0].set_title('BA: Bivalent vs Non-Bivalent', fontweight='bold'); axes[0].legend()
data = pd.DataFrame({'BA': list(biv_ba)+list(non_ba), 'Type': ['Bivalent']*len(biv_ba)+['Non-Bivalent']*len(non_ba)})
sns.boxplot(data=data, x='Type', y='BA', ax=axes[1])
axes[1].set_title('BA Comparison', fontweight='bold')
plt.tight_layout()
plt.savefig('figures/10_bivalent_domain_analysis.png', dpi=300)
plt.show()

## Chromatin State Classification

In [ ]:
def classify(marks):
    if 'H3K4me3' in marks and 'H3K27me3' in marks: return 'Bivalent'
    elif 'H3K4me3' in marks: return 'Active Promoter'
    elif 'H3K27ac' in marks and 'H3K4me1' in marks: return 'Active Enhancer'
    elif 'H3K27me3' in marks: return 'Repressed'
    elif 'H3K36me3' in marks: return 'Transcription'
    elif 'H3K9me3' in marks: return 'Heterochromatin'
    else: return 'Other'
reg_marks['state'] = reg_marks['mark_name'].apply(classify)
states = reg_marks['state'].value_counts()
print(states)
plt.figure(figsize=(10, 6))
states.plot(kind='barh', edgecolor='black')
plt.title('Chromatin States', fontweight='bold')
plt.savefig('figures/13_chromatin_state_classification.png', dpi=300)
plt.show()